# Lecture 11: Contact Dynamics

**Source span.** Printed pages 61-64; physical PDF pages 75-78 in `Lectures on Symplectic Geometry.pdf`. I inspected the local PDF text for this span before revising the notebook.

**Lecture goal.** Turn contact forms into dynamics: isolate the Reeb vector field, show why its flow preserves the contact form, build the symplectization layer, and connect periodic Reeb orbits to the Seifert-Weinstein questions.

A Reeb vector field is determined by two equations: `i_R d alpha = 0` and `alpha(R)=1`. In the standard `R^(2n+1)` model this gives vertical translation. On the odd sphere it gives the Hopf vector field, whose flow lines are circles. Symplectization then adds a real coordinate `tau` and converts contact data into the symplectic form `d(e^tau pi^* alpha)`.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np


def find_book_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "Lectures on Symplectic Geometry.pdf").exists():
            return candidate
    raise RuntimeError("Could not locate the Lectures-on-Symplectic-Geometry course root.")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, read_json, save_json, save_matplotlib

ARTIFACT_TOPIC = "lecture-11"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / ARTIFACT_TOPIC
(ARTIFACT_ROOT / "figures").mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / "checks").mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "font.size": 10})
print(f"Artifact topic: {ARTIFACT_TOPIC}")

## Translation Guide And Library Routing

| Source idea | Computational object | Check |
| --- | --- | --- |
| Reeb vector fields | solve two linear equations for `R` | `alpha(R)=1` and `i_R d alpha=0` |
| Reeb vector field in `R^(2n+1)` | `partial_z` for `alpha=sum x_i dy_i + dz` | flow is translation in `z` and preserves `alpha` |
| Hopf vector field | rotation in each `(x_i,y_i)` plane on the odd sphere | `alpha(R)=1`, `d alpha(R,v)=0` for tangent `v`, and orbits are periodic |
| Symplectization | matrix for `omega=d(e^tau pi^* alpha)` | determinant stays nonzero across `tau` layers |
| contact energy level | hypersurface carrying a contact form from ambient symplectic geometry | Reeb dynamics live on the energy level rather than in the full phase space |
| Seifert and Weinstein conjectures | periodic-orbit existence questions | distinguish arbitrary, volume-preserving, positive, and Reeb vector fields |

**Library routing.** `numpy` performs the Reeb and symplectization residual checks. `matplotlib` gives the vertical-flow, Hopf-fiber, and symplectization-layer views. `networkx` is used for the conjecture ledger because the source span is mostly implication and motivation rather than a single theorem proof.

## Visualization Storyboard

1. **Reeb flow in `R3` and on the sphere.** The `R3` standard contact form has vertical Reeb trajectories; the sphere example has Hopf circles. The learner should inspect that both are generated by the same two Reeb equations.
2. **Symplectization layer.** For `alpha = dz + x dy`, the symplectization matrix in coordinates `(tau,x,y,z)` is nondegenerate; the determinant grows with the `e^tau` factor.
3. **Seifert-Weinstein ledger.** The source moves from arbitrary vector fields on `S3`, to volume-preserving positive fields, to Reeb fields, and finally to periodic-orbit statements and open questions.

In [ ]:
# Reeb equations in R3: alpha = dz + x dy has R = partial_z.
reeb_r3 = np.array([0.0, 0.0, 1.0])
def alpha_r3(vector, x):
    return vector[2] + x * vector[1]

def dalpha_r3_on(R, V):
    # d alpha = dx wedge dy
    return R[0] * V[1] - R[1] * V[0]

sample_x = np.linspace(-1.5, 1.5, 9)
alpha_R_values = [alpha_r3(reeb_r3, x) for x in sample_x]
dalpha_R_residual = max(abs(dalpha_r3_on(reeb_r3, np.array([a, b, c]))) for a, b, c in [(1,0,0), (0,1,0), (0,0,1)])
assert max(abs(value - 1.0) for value in alpha_R_values) < 1e-12
assert dalpha_R_residual < 1e-12

# Hopf vector field on S3: R=2 sum (x_i partial_yi - y_i partial_xi).
def hopf_point(t, eta=0.72, phase=0.0):
    a = np.cos(eta)
    b = np.sin(eta)
    return np.array([a*np.cos(t), a*np.sin(t), b*np.cos(t+phase), b*np.sin(t+phase)])

def stereographic_s3_to_r3(p):
    denom = 1.0 - p[3]
    return p[:3] / denom

T = np.linspace(0, 2*np.pi, 360)
phis = [0.0, 1.4, 2.8]
hopf_curves = []
for phase in phis:
    curve4 = np.array([hopf_point(t, phase=phase) for t in T])
    assert np.max(np.abs(np.sum(curve4**2, axis=1) - 1.0)) < 1e-12
    hopf_curves.append(np.array([stereographic_s3_to_r3(p) for p in curve4]))

p = hopf_point(0.37, phase=0.8)
R = np.array([-2*p[1], 2*p[0], -2*p[3], 2*p[2]])
sigma_R = 0.5 * (p[0]*R[1] - p[1]*R[0] + p[2]*R[3] - p[3]*R[2])
tangent = np.array([p[2], -p[3], -p[0], p[1]])
tangent = tangent - np.dot(tangent, p) * p
dsigma_R_tangent = (R[0]*tangent[1] - R[1]*tangent[0]) + (R[2]*tangent[3] - R[3]*tangent[2])
assert abs(sigma_R - 1.0) < 1e-12
assert abs(dsigma_R_tangent) < 1e-12

fig = plt.figure(figsize=(12.5, 5.2))
ax = fig.add_subplot(1, 2, 1, projection="3d")
for x in [-1.0, 0.0, 1.0]:
    for y in [-0.7, 0.7]:
        z = np.linspace(-1.2, 1.2, 60)
        ax.plot(np.full_like(z, x), np.full_like(z, y), z, color="#1b9e77", lw=2)
        ax.quiver(x, y, 0.0, 0, 0, 0.45, color="#d95f02", linewidth=1.5)
ax.set_title("R3 model: Reeb flow translates in z")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.view_init(elev=22, azim=-52)

ax2 = fig.add_subplot(1, 2, 2, projection="3d")
for curve, color in zip(hopf_curves, ["#1b9e77", "#2c7fb8", "#d95f02"]):
    ax2.plot(curve[:,0], curve[:,1], curve[:,2], color=color, lw=2)
ax2.set_title("S3 model: Hopf/Reeb periodic orbits")
ax2.set_xlabel("stereo X")
ax2.set_ylabel("stereo Y")
ax2.set_zlabel("stereo Z")
ax2.view_init(elev=21, azim=-67)
fig.suptitle("Reeb vector fields are isolated by alpha(R)=1 and i_R d alpha=0")
reeb_flow_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "reeb-flow-r3-and-hopf-fibers.png")
plt.close(fig)
display_artifact(reeb_flow_path, width=760)
print({"r3_alpha_R": alpha_R_values[0], "hopf_sigma_R": sigma_R, "hopf_period": 2*np.pi})

In [ ]:
# Symplectization of alpha = dz + x dy: omega = d(e^tau alpha).
def symplectization_matrix(tau, x):
    e = np.exp(tau)
    M = np.zeros((4, 4))  # coordinates tau, x, y, z
    # e d tau wedge dz
    M[0, 3] = e
    M[3, 0] = -e
    # e x d tau wedge dy
    M[0, 2] = e * x
    M[2, 0] = -e * x
    # e dx wedge dy
    M[1, 2] = e
    M[2, 1] = -e
    return M

taus = np.linspace(-1.2, 1.2, 80)
layer_dets = np.array([np.linalg.det(symplectization_matrix(tau, 0.6)) for tau in taus])
min_layer_det = float(np.min(np.abs(layer_dets)))
assert min_layer_det > 0.0

matrix_sample = symplectization_matrix(0.25, 0.6)
skew_residual = float(np.linalg.norm(matrix_sample + matrix_sample.T))
assert skew_residual < 1e-12

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
im = axes[0].imshow(matrix_sample, cmap="coolwarm")
axes[0].set_title("omega matrix at tau=0.25, x=0.6")
axes[0].set_xticks(range(4), labels=["tau", "x", "y", "z"])
axes[0].set_yticks(range(4), labels=["tau", "x", "y", "z"])
for (i, j), value in np.ndenumerate(matrix_sample):
    if abs(value) > 1e-9:
        axes[0].text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=axes[0], fraction=0.046)
axes[1].plot(taus, np.abs(layer_dets), color="#1b9e77", lw=2)
axes[1].set_yscale("log")
axes[1].set_title("nondegeneracy across symplectization layers")
axes[1].set_xlabel("tau")
axes[1].set_ylabel("|det omega_tau|")
axes[1].axhline(min_layer_det, color="#d95f02", ls="--", lw=1)
fig.suptitle("Symplectization: d(e^tau pi^* alpha) is symplectic")
symplectization_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "symplectization-layer-symplectic-matrix.png")
plt.close(fig)
display_artifact(symplectization_path, width=760)
print({"min_layer_det": min_layer_det, "skew_residual": skew_residual})

In [ ]:
# Periodic-orbit motivation ledger: Seifert -> positive fields -> Reeb -> Weinstein conjecture.
G = nx.DiGraph()
G.add_edges_from([
    ("nowhere-vanishing vector field on S3", "Seifert question"),
    ("Seifert question", "smooth counterexamples exist"),
    ("volume-preserving field", "i_v gamma is closed"),
    ("H2(S3)=0", "i_v gamma = d alpha"),
    ("positive field: alpha(v)>0", "renormalize to R=v/alpha(v)"),
    ("renormalize to R=v/alpha(v)", "i_R d alpha=0 and alpha(R)=1"),
    ("i_R d alpha=0 and alpha(R)=1", "Reeb vector field"),
    ("Reeb vector field", "Weinstein conjecture: periodic orbit"),
    ("S3 / pi2 nonzero / overtwisted", "Viterbo-Hofer theorem cases"),
    ("Viterbo-Hofer theorem cases", "periodic orbit guaranteed"),
    ("contact energy level", "Reeb dynamics on hypersurface"),
    ("Reeb dynamics on hypersurface", "periodic orbit question"),
    ("periodic orbit question", "how many / linking / unknotting"),
])
pos = {
    "nowhere-vanishing vector field on S3": (0, 2.2),
    "Seifert question": (1.7, 2.2),
    "smooth counterexamples exist": (3.4, 2.2),
    "volume-preserving field": (0, 1.0),
    "i_v gamma is closed": (1.7, 1.0),
    "H2(S3)=0": (0, -0.1),
    "i_v gamma = d alpha": (1.7, -0.1),
    "positive field: alpha(v)>0": (3.4, 0.45),
    "renormalize to R=v/alpha(v)": (5.1, 0.45),
    "i_R d alpha=0 and alpha(R)=1": (6.8, 0.45),
    "Reeb vector field": (8.5, 0.45),
    "Weinstein conjecture: periodic orbit": (10.2, 0.45),
    "S3 / pi2 nonzero / overtwisted": (6.8, -0.85),
    "Viterbo-Hofer theorem cases": (8.5, -0.85),
    "periodic orbit guaranteed": (10.2, -0.85),
    "contact energy level": (3.4, -1.7),
    "Reeb dynamics on hypersurface": (5.1, -1.7),
    "periodic orbit question": (6.8, -1.7),
    "how many / linking / unknotting": (8.5, -1.7),
}
fig, ax = plt.subplots(figsize=(14, 5.8))
colors = []
for node in G.nodes:
    if "Reeb" in node or "Weinstein" in node:
        colors.append("#b2df8a")
    elif "counter" in node or "Seifert" in node:
        colors.append("#fdbf6f")
    else:
        colors.append("#a6cee3")
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=14, edge_color="0.36")
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors, node_size=2050, edgecolors="0.25", linewidths=0.8)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=7)
ax.set_axis_off()
ax.set_title("From Seifert's question to Weinstein's Reeb periodic-orbit conjecture")
conjecture_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "weinstein-conjecture-periodic-orbit-ledger.png")
plt.close(fig)
display_artifact(conjecture_path, width=760)
print({"ledger_nodes": G.number_of_nodes(), "ledger_edges": G.number_of_edges()})

## Reading The Visuals

The first figure separates the two model Reeb flows. In `R3`, the equations force `R=partial_z`, so the flow preserves the contact form by translation. On the odd sphere, the Reeb field rotates every complex coordinate at the same angular speed; after stereographic projection the Hopf fibers appear as closed curves, making periodic orbit behavior visible.

The symplectization panel is the algebraic check behind `omega=d(e^tau pi^* alpha)`. The matrix is skew and its determinant is nonzero across sampled `tau` layers, so the extra real direction has converted the contact form into a symplectic form.

The conjecture ledger keeps the history straight. Seifert's question has counterexamples for arbitrary fields. The Weinstein conjecture concerns Reeb fields of contact forms, a much more rigid class; the source records the Viterbo-Hofer theorem cases and the open questions about number and shape of periodic orbits.

In [ ]:
residuals = {
    "r3_alpha_R_values_max_error": max(abs(value - 1.0) for value in alpha_R_values),
    "r3_iR_dalpha_residual": dalpha_R_residual,
    "hopf_alpha_R_minus_one": float(abs(sigma_R - 1.0)),
    "hopf_iR_dalpha_tangent_residual": float(abs(dsigma_R_tangent)),
    "hopf_orbit_period_parameter": float(2*np.pi),
    "symplectization_min_abs_det": min_layer_det,
    "symplectization_skew_residual": skew_residual,
    "conjecture_ledger_nodes": G.number_of_nodes(),
    "conjecture_ledger_edges": G.number_of_edges(),
}
residual_path = save_json(residuals, ARTIFACT_TOPIC, "checks", "contact-dynamics-residuals.json")

source_span = {
    "title": "Contact Dynamics",
    "printed_pages": "61-64",
    "physical_pdf_pages": "75-78",
    "source_file": "Lectures on Symplectic Geometry.pdf",
    "sections": [
        "Reeb vector fields",
        "Symplectization",
        "Seifert and Weinstein conjectures",
    ],
    "concepts": [
        "Reeb vector field",
        "symplectization",
        "periodic orbit",
        "contact energy level",
        "Hopf vector field",
        "Weinstein conjecture",
    ],
    "copyright_note": "Source pages were used for terminology and coverage; notebook prose, visuals, and code are original and no page crops or long exercise text are copied.",
}
source_path = save_json(source_span, ARTIFACT_TOPIC, "checks", "source-span.json")

storyboard = {
    "chapter_goal": "Make Reeb dynamics, symplectization, and the periodic-orbit motivation inspectable with flow and matrix checks.",
    "source_span_read": source_span,
    "library_routing": [
        {"concept": "Reeb vector field", "representation": "flow lines and equation residuals", "library": "numpy + matplotlib", "why": "Reeb fields are defined by two checkable equations"},
        {"concept": "symplectization", "representation": "skew matrix and determinant layer plot", "library": "numpy + matplotlib", "why": "nondegeneracy is a matrix determinant in a local contact chart"},
        {"concept": "periodic orbit", "representation": "Hopf fiber projection", "library": "numpy + matplotlib", "why": "the sphere example turns Reeb flow into visible closed curves"},
        {"concept": "Seifert and Weinstein conjectures", "representation": "dependency ledger", "library": "networkx", "why": "the source span is a chain of motivation and theorem cases"},
    ],
    "visual_sequence": [
        {"concept": "Reeb flow helix and Hopf fibers", "artifact": "artifacts/lecture-11/figures/reeb-flow-r3-and-hopf-fibers.png", "inspection_target": "Reeb equations produce vertical and closed Hopf flows"},
        {"concept": "symplectization layer", "artifact": "artifacts/lecture-11/figures/symplectization-layer-symplectic-matrix.png", "inspection_target": "d(e^tau alpha) is skew and nondegenerate"},
        {"concept": "periodic orbit conjecture ledger", "artifact": "artifacts/lecture-11/figures/weinstein-conjecture-periodic-orbit-ledger.png", "inspection_target": "Reeb fields are the rigid class in the Weinstein conjecture"},
    ],
    "computational_checks": residuals,
}
storyboard_path = save_json(storyboard, ARTIFACT_TOPIC, "checks", "visual-storyboard.json")

final_sanity = {
    "artifacts": [
        str(reeb_flow_path.relative_to(BOOK_ROOT)),
        str(symplectization_path.relative_to(BOOK_ROOT)),
        str(conjecture_path.relative_to(BOOK_ROOT)),
        str(residual_path.relative_to(BOOK_ROOT)),
        str(source_path.relative_to(BOOK_ROOT)),
        str(storyboard_path.relative_to(BOOK_ROOT)),
    ],
    "assertions": {
        "r3_reeb_alpha_R_is_one": bool(max(abs(value - 1.0) for value in alpha_R_values) < 1e-12),
        "r3_reeb_contracts_dalpha_to_zero": bool(dalpha_R_residual < 1e-12),
        "hopf_alpha_R_is_one": bool(abs(sigma_R - 1.0) < 1e-12),
        "hopf_iR_dalpha_vanishes_on_tangent": bool(abs(dsigma_R_tangent) < 1e-12),
        "symplectization_matrix_is_skew": bool(skew_residual < 1e-12),
        "symplectization_is_nondegenerate": bool(min_layer_det > 0.0),
        "periodic_orbit_ledger_is_populated": bool(G.number_of_edges() >= 10),
    },
}
final_path = save_json(final_sanity, ARTIFACT_TOPIC, "checks", "final-sanity.json")
print(json.dumps(final_sanity, indent=2))

In [ ]:
final_sanity = read_json(ARTIFACT_ROOT / "checks" / "final-sanity.json")
assert all(final_sanity["assertions"].values())
for relative in final_sanity["artifacts"]:
    path = BOOK_ROOT / relative
    assert path.exists(), relative
    assert path.stat().st_size > 100, relative

for relative in final_sanity["artifacts"]:
    path = BOOK_ROOT / relative
    if path.suffix.lower() in {".png", ".html", ".svg"}:
        display_artifact(path, width=760, height=430 if path.suffix == ".html" else None)

print("Lecture 11 checks passed; artifacts are present and nonempty.")

## Takeaways

- The Reeb vector field is uniquely determined by `alpha(R)=1` and `i_R d alpha=0`; its flow preserves `alpha` by Cartan's formula.
- In standard contact Euclidean space, Reeb flow is translation in the `z` direction. On the odd sphere, the Reeb/Hopf flow has closed circular orbits.
- Symplectization adds a real coordinate and turns contact data into a symplectic form `d(e^tau pi^* alpha)`.
- The Weinstein conjecture is not Seifert's question for arbitrary vector fields; it is a periodic-orbit question for the rigid Reeb vector fields of contact forms.
- Contact energy levels are where symplectic and contact dynamics meet: the ambient symplectic geometry supplies a hypersurface contact form, and Reeb dynamics studies the induced flow.